In [17]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams["font.family"] = "DejaVu Sans"

offset = 10
TITLE_FS = 30 + offset
YLABEL_FS = 24 + offset
TICK_FS = 24 + offset
BARLABEL_FS = 15 + offset

def color_map(values):
    values = np.array(values)
    norm = mcolors.Normalize(vmin=values.min(), vmax=values.max())
    cmap = matplotlib.colormaps["RdYlGn"]
    return [cmap(norm(v)) for v in values]

def bar_chart(ax, labels, values, title, ylabel, ymax, label_offset, norm):
    colors = color_map(values, norm)
    bars = ax.bar(labels, values, color=colors, width=0.72)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + label_offset,
                f"{v:.1f}", ha="center", va="bottom", fontsize=BARLABEL_FS)
    ax.set_ylabel(ylabel, fontsize=YLABEL_FS)
    ax.set_title(title, fontsize=TITLE_FS, pad=20)
    ax.set_ylim(0, ymax)
    ax.set_yticks(np.arange(0, ymax + 0.01, 10))
    ax.set_yticklabels([f"{int(t)}" for t in np.arange(0, ymax + 0.01, 10)],
                       fontsize=TICK_FS)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=TICK_FS)

mmmlu_labels = ["ZH", "ES", "IT", "PT", "DE", "FR",
                "ID", "JA", "KO", "AR", "HI", "BN",
                "SW", "YO"]
mmmlu_vals = [67.8, 67.7, 66.5, 66.0, 65.8, 65.1, 63.9, 62.7, 57.4, 55.6,
              47.2, 46.2, 36.9, 32.7]

hotpot_labels = ["EN", "CN", "RU", "AR"]
hotpot_vals = [84.3, 73.4, 65.7, 63.6]

nemo_labels = ["IT", "EN", "JA", "ES", "DE", "FR"]
nemo_vals = [54.6, 50.0, 49.8, 47.6, 47.5, 34.9]

all_vals = mmmlu_vals + hotpot_vals + nemo_vals
shared_norm = mcolors.Normalize(vmin=min(all_vals), vmax=max(all_vals))
# → vmin=32.7 (Yoruba, MMMLU), vmax=84.3 (English, HotPotQA)

def color_map(values, norm):
    cmap = matplotlib.colormaps["RdYlGn"]
    return [cmap(norm(v)) for v in values]

out = "benchmark_figures.pdf"
with PdfPages(out) as pdf:
    fig, ax = plt.subplots(figsize=(16.5, 9.5))
    bar_chart(ax, mmmlu_labels, mmmlu_vals, "MMMLU", "Pass@5 Rate (%)", 78, 0.8, norm=shared_norm)
    fig.tight_layout()
    pdf.savefig(fig); plt.close(fig)

    fig, ax = plt.subplots(figsize=(13.5, 9.5))
    bar_chart(ax, hotpot_labels, hotpot_vals, "HotPotQA", "Pass@5 Rate (%)", 92, 1.2, norm=shared_norm)
    fig.tight_layout()
    pdf.savefig(fig); plt.close(fig)

    fig, ax = plt.subplots(figsize=(13.5, 9.5))
    bar_chart(ax, nemo_labels, nemo_vals, "Nemotron", "Pass@5 Rate (%)", 62, 0.8, norm=shared_norm)
    fig.tight_layout()
    pdf.savefig(fig); plt.close(fig)

In [25]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib
import numpy as np
from matplotlib.patches import Patch
from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams["font.family"] = "DejaVu Sans"


offset = 10
TITLE_FS = 30 + offset
YLABEL_FS = 24 + offset
TICK_FS = 24 + offset
BARLABEL_FS = 15 + offset
LEGEND_FS = 15 + offset

SHARED_NORM = mcolors.Normalize(vmin=32.7, vmax=84.3)
CMAP = matplotlib.colormaps["RdYlGn"]

languages = ["EN", "CN", "RU", "AR"]
qwen_vals = [84.3, 73.4, 65.7, 63.6]   # Qwen2.5-7B-Instruct
aya_vals  = [70.6, 74.4, 73.9, 69.3]   # Aya-23-8B

x = np.arange(len(languages))
w = 0.38

fig, ax = plt.subplots(figsize=(14.5, 9.5))

qwen_colors = [CMAP(SHARED_NORM(v)) for v in qwen_vals]
aya_colors  = [CMAP(SHARED_NORM(v)) for v in aya_vals]

b1 = ax.bar(x - w/2, qwen_vals, w, color=qwen_colors, linewidth=0.8)
b2 = ax.bar(x + w/2, aya_vals, w, color=aya_colors, linewidth=0.8, hatch=".")

for bars, vals in [(b1, qwen_vals), (b2, aya_vals)]:
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 1.0, f"{v:.1f}",
                ha="center", va="bottom", fontsize=BARLABEL_FS)

ax.set_ylabel("Pass@5 Rate (%)", fontsize=YLABEL_FS)
ax.set_title("HotPotQA", fontsize=TITLE_FS, pad=20)
ax.set_ylim(0, 95)
ax.set_yticks(np.arange(0, 95.01, 10))
ax.set_yticklabels([f"{int(t)}" for t in np.arange(0, 95.01, 10)], fontsize=TICK_FS)
ax.set_xticks(x)
ax.set_xticklabels(languages, rotation=45, ha="right", fontsize=TICK_FS)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = [
    Patch(facecolor="0.75", edgecolor="black", label="Qwen2.5-7B-Instruct"),
    Patch(facecolor="0.75", edgecolor="black", hatch="//", label="Aya-23-8B"),
]
ax.legend(handles=legend_handles, fontsize=LEGEND_FS, frameon=False, loc="upper right")

fig.tight_layout()
with PdfPages("hotpotqa_qwen_vs_aya.pdf") as pdf:
    pdf.savefig(fig)
plt.close(fig)